In [4]:
# Image preprocessing (.jpg)
# -----------------------------

import cv2
import numpy as np
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
import os
import glob

input_dir = Path(r"C:\Users\LENOVO\Desktop\ImpPic\_1All_implant(NoProsth)")
output_dir = Path(r"C:\Users\LENOVO\Desktop\ImpPic\_2EnhancePA(NoProsth)")
augoutput_dir = Path(r"C:\Users\LENOVO\Desktop\ImpPic\_3AugmentPA(NoProsth)")
UnetTrain_dir = Path(r"C:\Users\LENOVO\Desktop\ImpPic\_4UnetGroup\Train")
UnetVal_dir = Path(r"C:\Users\LENOVO\Desktop\ImpPic\_4UnetGroup\Val")
UnetTest_dir = Path(r"C:\Users\LENOVO\Desktop\ImpPic\_4UnetGroup\Test")
UnetTrain_mask_dir = Path(r"C:\Users\LENOVO\Desktop\ImpPic\_4UnetGroup\Train_mask")
UnetVal_mask_dir = Path(r"C:\Users\LENOVO\Desktop\ImpPic\_4UnetGroup\Val_mask")
UnetTest_mask_dir = Path(r"C:\Users\LENOVO\Desktop\ImpPic\_4UnetGroup\Test_mask")
# -----------------------------
#Resize+padding 
def resize_with_padding(img_gray, out_size=512, pad_value=0):
    h, w = img_gray.shape[:2]
    scale = out_size / max(h, w)
    resized = cv2.resize(img_gray, (int(round(w * scale)), int(round(h * scale))), 
                         interpolation=cv2.INTER_AREA)

    EnhancePA = np.full((out_size, out_size), pad_value, dtype=resized.dtype)
    new_h, new_w = resized.shape[:2]
    EnhancePA[(out_size-new_h)//2 : (out_size-new_h)//2 + new_h,
           (out_size-new_w)//2 : (out_size-new_w)//2 + new_w] = resized
    return EnhancePA

#EnhancePA 
def preprocess(img_bgr, out_size=512, use_negative=False):

    # 1) grayscale
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    
    # 2) gamma correction
    x = gray.astype(np.float32) / 255.0
    x = np.power(x, 0.8)          # gamma = 0.8
    gray = (x * 255.0).clip(0, 255).astype(np.uint8)

    # 3) CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    # 4) Negative film
    if use_negative:
        gray = 255 - gray
    return gray

# Augmentation
def augment():
    rotations = {
        0: None,
        90: cv2.ROTATE_90_CLOCKWISE,
        180: cv2.ROTATE_180,
        270: cv2.ROTATE_90_COUNTERCLOCKWISE
    }

    for p in output_dir.glob("*.jpg"):
        img = cv2.imread(str(p))
        variants = {"ori": img, "Hflip": cv2.flip(img, 1)}
        for vname, vimg in variants.items():
            for ang, code in rotations.items():
                out = vimg if code is None else cv2.rotate(vimg, code)
                out_name = f"{p.stem}_{vname}_r{ang}.jpg"
                cv2.imwrite(str(augoutput_dir / out_name), out) 

# Split group
def splitGroup(augoutput_dir):
    image_paths = sorted(glob.glob(os.path.join(str(augoutput_dir), "*.jpg")))
    
    # 80% (train+val) และ 20% (test)
    Train_Val_paths, Test_paths = train_test_split(image_paths, test_size=0.2, shuffle=True)
    
    # จาก 80% → train 60% และ val 20% (25% ของ 80% = 20% ของทั้งหมด)
    Train_paths, Val_paths = train_test_split(Train_Val_paths, test_size=0.25, shuffle=True)
    return Train_paths, Val_paths, Test_paths

# Create group
def copyFileToGroup(imgpaths, group):
    count1 = 0
    for doner_path in imgpaths:
        imgname = os.path.basename(doner_path)
        recipient_path = os.path.join(group, imgname)
        shutil.copy(doner_path, recipient_path)
        count1 += 1
# -----------------------------

count2 = 0
for img_path in input_dir.glob("*.jpg"):
    img = cv2.imread(str(img_path))
    out = preprocess(img, out_size=512, use_negative=True)
    cv2.imwrite(str(output_dir / img_path.name), out)
    count2 += 1

augment()

Train_paths, Val_paths, Test_paths = splitGroup(augoutput_dir)

copyFileToGroup(Train_paths, UnetTrain_dir)
copyFileToGroup(Val_paths,   UnetVal_dir)
copyFileToGroup(Test_paths,  UnetTest_dir)

print(f"split -> train: {len(Train_paths)}, val: {len(Val_paths)}, test: {len(Test_paths)}")
print("Finished")

split -> train: 3504, val: 1168, test: 1168
Finished


In [8]:
# Create mask (Labelme)
# -----------------------------

import cv2
import numpy as np
from pathlib import Path
import json

base = Path(r"C:\Users\LENOVO\Desktop\ImpPic\_4UnetGroup")

for split in ["Train", "Val", "Test"]:
    for fjson in (base / split).glob("*.json"):
        d = json.loads(fjson.read_text(encoding="utf-8"))
        mask = np.zeros((d["imageHeight"], d["imageWidth"]), np.uint8)

        for s in d.get("shapes", []):
            cv2.fillPoly(mask, [np.array(s["points"], dtype=np.int32)], 255)

        out_path = base / f"{split}_mask" / f"{fjson.stem}_mask.jpg"
        cv2.imwrite(str(out_path), mask)

print("Finished")

Finished
